# Find a Rubin object by its Rubin `provid` or its MPC designation

Many objects observed by Rubin got a new provisional designation (`provid`, e.g. `2025 HB67`), and the MPC later
linked them to an object that already existed with another designation (e.g. `2022 UK169`).
In `obs_final_elements_geometry_H.parquet` every detection has **both** ids:

- `provid`: the designation reported by Rubin
- `mpc_designation`: the designation of the object in the MPC catalogue

and `data/mpc_designations.csv` is the table of equivalences `provid -> mpc_designation`.

In [9]:
import pandas as pd

data_dir = 'data/'

obs = pd.read_parquet(data_dir + 'obs_final_elements_geometry_H.parquet', columns=['provid', 'mpc_designation', 'H'])

ids = pd.read_csv(data_dir + 'mpc_designations.csv')

def find_objects(object_ids):
    # MPC designation of each id (an id can be a Rubin provid or already an MPC designation)
    query = pd.DataFrame({'search_id': object_ids})
    query['mpc_designation'] = query['search_id'].map(ids.set_index('provid')['mpc_designation'])
    query['mpc_designation'] = query['mpc_designation'].fillna(query['search_id'].where(query['search_id'].isin(ids['mpc_designation'])))

    # All detections of these objects, and H per object
    det = obs[obs['mpc_designation'].isin(query['mpc_designation'])]
    per_object = det.groupby('mpc_designation').agg(provids=('provid', lambda p: ', '.join(sorted(p.unique()))),
                                                    H=('H', 'median'), H_std=('H', 'std'), n_obs=('H', 'size'))
    return query.merge(per_object, on='mpc_designation', how='left').round(2)

find_objects(['2025 HB67', '2022 UK169', '2026 DO29', '2015 CY86', '2025 JQ7'])

,search_id,mpc_designation,provids,H,H_std,n_obs
0,2025 HB67,2022 UK169,2025 HB67,18.59,0.31,296
1,2022 UK169,2022 UK169,2025 HB67,18.59,0.31,296
2,2026 DO29,2015 CY86,2026 DO29,18.09,0.29,741
3,2015 CY86,2015 CY86,2026 DO29,18.09,0.29,741
4,2025 JQ7,2025 NF311,"2025 JQ7, 2025 NF311",19.23,0.31,129


## Objects with two provids in the Rubin catalogue

How many objects are like `2025 JQ7`? In the Rubin catalogue (`obs_sbn_X05_full_mpc_desig.parquet`) each detection
keeps the `provid` it was reported with, and `mpc_designation` is the object it belongs to according to the MPC
(found in MPCORB by number, main designation or other designations, or else with the MPC API; see `add_mpc_designation.ipynb`).
An MPC object with more than one `provid` was reported by Rubin as separate objects.

In [10]:
# Rubin catalogue with the MPC designation of each detection
rubin = pd.read_parquet(data_dir + 'obs_sbn_X05_full_mpc_desig.parquet',
                        columns=['provid', 'mpc_designation', 'match_by', 'obstime'])
rubin = rubin.dropna(subset=['provid', 'mpc_designation'])

# One row per (MPC object, Rubin provid)
per_provid = (rubin.groupby(['mpc_designation', 'provid'])
              .agg(match_by=('match_by', 'first'), n_obs=('obstime', 'size'),
                   first_obs=('obstime', 'min'), last_obs=('obstime', 'max'))
              .reset_index())

# MPC objects reported by Rubin with more than one provid
n_provids = per_provid.groupby('mpc_designation')['provid'].transform('nunique')
split = per_provid[n_provids > 1].sort_values(['mpc_designation', 'first_obs'])
print(f"{split['mpc_designation'].nunique():,} MPC objects appear in the Rubin catalogue with more than one provid "
      f"({len(split):,} provids)")

# One row per object: its provids in the order Rubin observed them
split_objects = (split.groupby('mpc_designation', sort=False)
                 .agg(provids=('provid', ', '.join), n_obs=('n_obs', lambda n: ', '.join(map(str, n))),
                      n_obs_total=('n_obs', 'sum'), first_obs=('first_obs', 'min'), last_obs=('last_obs', 'max'))
                 .reset_index())
split_objects

2,989 MPC objects appear in the Rubin catalogue with more than one provid (5,978 provids)


,mpc_designation,provids,n_obs,n_obs_total,first_obs,last_obs
0,(831800) 2009 WC150,"2009 WC150, 2025 MN83","381, 28",409,2025-04-25 02:51:52.479,2025-05-04 04:13:09.801
1,(831814) 2009 WY177,"2009 WY177, 2025 ML51","55, 11",66,2025-04-22 02:37:21.011,2025-05-04 03:18:57.060
2,(835088) 2011 BY148,"2025 MS60, 2011 BY148","12, 88",100,2025-04-22 02:43:22.655,2025-05-04 03:40:31.048
3,(840358) 2015 AF100,"2015 AF100, 2025 MH16","242, 9",251,2025-04-28 01:21:54.393,2025-05-04 04:42:15.272
4,(881568) 2015 KL461,"2025 MR86, 2015 KL461","10, 415",425,2025-04-25 02:52:37.015,2025-05-04 03:46:55.649
...,...,...,...,...,...,...
2984,2025 VG37,"2025 OP666, 2025 VG37","6, 6",12,2025-07-18 06:38:01.763,2025-11-22 02:10:53.449
2985,2026 AM24,"2026 AM24, 2026 GW14","43, 7",50,2025-12-15 07:40:39.472,2026-04-12 23:47:48.080
2986,2026 DT28,"2026 DJ35, 2026 DT28","177, 126",303,2026-02-19 05:56:24.776,2026-02-27 05:42:44.288
2987,2026 ET58,"2025 KG47, 2026 ET58","157, 21",178,2025-05-24 00:39:02.266,2026-03-19 06:02:51.382


In [ ]:
out = data_dir + 'rubin_objects_two_provids.csv'
split_objects.to_csv(out, index=False)
print(f'Saved {len(split_objects):,} objects to {out}')